In [1]:
import joblib
from scipy.sparse import load_npz
import os


In [2]:
# --- Load the processed data from the files ---
output_dir = 'processed_data'

# Load the TF-IDF matrices
X_train_tfidf = load_npz(os.path.join(output_dir, 'X_train_tfidf.npz'))
X_test_tfidf = load_npz(os.path.join(output_dir, 'X_test_tfidf.npz'))

# Load the labels
y_train = joblib.load(os.path.join(output_dir, 'y_train.joblib'))
y_test = joblib.load(os.path.join(output_dir, 'y_test.joblib'))

# Load the vectorizer
tfidf_vectorizer = joblib.load(os.path.join(output_dir, 'tfidf_vectorizer.joblib'))

print("All data loaded successfully!")
print(f"Shape of loaded X_train_tfidf: {X_train_tfidf.shape}")
print(f"Shape of loaded y_train: {y_train.shape}")

All data loaded successfully!
Shape of loaded X_train_tfidf: (46909, 5000)
Shape of loaded y_train: (46909,)


### HP Tune

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

# --- 1. Train and Evaluate Logistic Regression ---
print("--- Training Logistic Regression Model ---")
# We increase max_iter to ensure the model converges
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train_tfidf, y_train)

# Make predictions on the test set
y_pred_log_reg = log_reg.predict(X_test_tfidf)

# Evaluate the model
accuracy_lr = accuracy_score(y_test, y_pred_log_reg)
print(f"\nLogistic Regression Accuracy: {accuracy_lr:.4f}")
print("Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_log_reg))


--- Training Logistic Regression Model ---

Logistic Regression Accuracy: 0.8938
Logistic Regression Classification Report:
                         precision    recall  f1-score   support

Bank account or service       0.87      0.82      0.84      1142
            Credit card       0.85      0.83      0.84      1586
       Credit reporting       0.89      0.87      0.88      2505
        Debt collection       0.88      0.91      0.90      3511
               Mortgage       0.95      0.96      0.95      2984

               accuracy                           0.89     11728
              macro avg       0.89      0.88      0.88     11728
           weighted avg       0.89      0.89      0.89     11728



In [4]:
# --- 2. Train and Evaluate Multinomial Naive Bayes ---
print("\n--- Training Multinomial Naive Bayes Model ---")
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

# Make predictions on the test set
y_pred_nb = nb_model.predict(X_test_tfidf)

# Evaluate the model
accuracy_nb = accuracy_score(y_test, y_pred_nb)
print(f"\nNaive Bayes Accuracy: {accuracy_nb:.4f}")
print("Naive Bayes Classification Report:")
print(classification_report(y_test, y_pred_nb))


--- Training Multinomial Naive Bayes Model ---

Naive Bayes Accuracy: 0.8586
Naive Bayes Classification Report:
                         precision    recall  f1-score   support

Bank account or service       0.91      0.73      0.81      1142
            Credit card       0.82      0.76      0.79      1586
       Credit reporting       0.82      0.84      0.83      2505
        Debt collection       0.85      0.88      0.86      3511
               Mortgage       0.91      0.95      0.93      2984

               accuracy                           0.86     11728
              macro avg       0.86      0.83      0.84     11728
           weighted avg       0.86      0.86      0.86     11728



In [5]:
from sklearn.model_selection import GridSearchCV

print("--- Starting Hyperparameter Tuning for Logistic Regression ---")

# Define the model we want to tune
log_reg_model = LogisticRegression(random_state=42, max_iter=2000)

# Define the grid of parameters to search
# 'C' is the inverse of regularization strength; smaller values specify stronger regularization.
param_grid = {
    'C': [0.1, 1, 10, 100]
}

# Set up the grid search with 5-fold cross-validation
grid_search = GridSearchCV(
    estimator=log_reg_model,
    param_grid=param_grid,
    scoring='f1_macro',  # Use F1-score as it's good for imbalanced classes
    cv=5,                # 5-fold cross-validation
    n_jobs=-1,           # Use all available CPU cores
    verbose=2            # Show progress
)

# Fit the grid search to the data
grid_search.fit(X_train_tfidf, y_train)

# --- Evaluate the Best Model ---
print("\n--- Tuning Complete ---")
print(f"Best Parameters Found: {grid_search.best_params_}")
print(f"Best Cross-Validation F1-Score (Macro): {grid_search.best_score_:.4f}")

# Get the best model from the grid search
best_log_reg = grid_search.best_estimator_

# Make predictions on the test set using the tuned model
y_pred_tuned = best_log_reg.predict(X_test_tfidf)

# Evaluate the final tuned model
print("\n--- Final Tuned Model Evaluation ---")
accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
print(f"Tuned Logistic Regression Accuracy: {accuracy_tuned:.4f}")
print("Tuned Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_tuned))

--- Starting Hyperparameter Tuning for Logistic Regression ---
Fitting 5 folds for each of 4 candidates, totalling 20 fits

--- Tuning Complete ---
Best Parameters Found: {'C': 1}
Best Cross-Validation F1-Score (Macro): 0.8789

--- Final Tuned Model Evaluation ---
Tuned Logistic Regression Accuracy: 0.8938
Tuned Logistic Regression Classification Report:
                         precision    recall  f1-score   support

Bank account or service       0.87      0.82      0.84      1142
            Credit card       0.85      0.83      0.84      1586
       Credit reporting       0.89      0.87      0.88      2505
        Debt collection       0.88      0.91      0.90      3511
               Mortgage       0.95      0.96      0.95      2984

               accuracy                           0.89     11728
              macro avg       0.89      0.88      0.88     11728
           weighted avg       0.89      0.89      0.89     11728



## Pipeline

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score

In [8]:
from sklearn.model_selection import train_test_split
import pandas as pd

In [19]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split


In [20]:
# --- Download NLTK resources (only needs to be done once) ---
print("Downloading NLTK resources...")
nltk.download('stopwords')
nltk.download('wordnet')
print("Downloads complete.")

Downloads complete.


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\arsha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\arsha\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [21]:

# --- 1. Text Cleaning and Lemmatization ---
print("\nPreprocessing text data...")

# Initialize the lemmatizer and stopwords list
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))


Preprocessing text data...


In [22]:
def preprocess_text(text):
    # Remove punctuation and numbers
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    # Convert to lowercase
    text = text.lower()
    # Tokenize (split into words)
    words = text.split()
    # Lemmatize and remove stopwords
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(words)

In [23]:
import pandas as pd
df = pd.read_csv(r'C:\Users\arsha\Downloads\suppor-ticket-classifier\data\complaints.csv')

In [24]:
# Apply the preprocessing function to the 'Description' column
# This might take a minute or two to run on all 58k rows
df['Cleaned Description'] = df['Description'].apply(preprocess_text)

print("Text preprocessing complete.")
print("\nOriginal vs. Cleaned Description:")
display(df[['Description', 'Cleaned Description']].head())

Text preprocessing complete.

Original vs. Cleaned Description:


,Description,Cleaned Description
0,XXXX has claimed I owe them {$27.00} for XXXX ...,xxxx claimed owe xxxx year despite proof payme...
1,In XX/XX/XXXX my wages that I earned at my job...,xx xx xxxx wage earned job decreased almost ha...
2,I have an open and current mortgage with Chase...,open current mortgage chase bank xxxx chase re...
3,XXXX was submitted XX/XX/XXXX. At the time I s...,xxxx submitted xx xx xxxx time submitted compl...
4,Experian is reporting my OPEN and CURRENT Mort...,experian reporting open current mortgage loan ...


In [25]:

# We need to reload our raw text data for the pipeline
X_train, X_test, y_train, y_test = train_test_split(df['Cleaned Description'], df['Category'],
                                                    test_size=0.2, random_state=42, stratify=df['Category'])


In [26]:

# --- 1. Create a Scikit-Learn Pipeline ---
# The pipeline bundles preprocessing (TF-IDF) and the model (Logistic Regression) into one object.
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000)),
    ('classifier', LogisticRegression(random_state=42, max_iter=2000))
])

In [27]:
# --- 2. Define the Parameter Grid for the Pipeline ---
# When tuning a pipeline, you must prefix parameter names with the step name (e.g., 'classifier__C').
param_grid = {
    'tfidf__max_df': [0.75, 1.0],         # Ignore words that appear in > 75% or 100% of documents
    'classifier__C': [0.1, 1, 10]         # Regularization strength for the classifier
}

In [28]:
# --- 3. Perform Grid Search on the Pipeline ---
print("--- Starting Hyperparameter Tuning for the Pipeline ---")
grid_search_pipeline = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=3,  # Using 3 folds for speed, 5 is also common
    n_jobs=-1,
    verbose=2
)

--- Starting Hyperparameter Tuning for the Pipeline ---


In [29]:
# Fit the grid search to the raw training text data
grid_search_pipeline.fit(X_train, y_train)

# --- 4. Evaluate the Best Pipeline ---
print("\n--- Tuning Complete ---")
print(f"Best Parameters Found: {grid_search_pipeline.best_params_}")
print(f"Best Cross-Validation F1-Score (Macro): {grid_search_pipeline.best_score_:.4f}")


Fitting 3 folds for each of 6 candidates, totalling 18 fits

--- Tuning Complete ---
Best Parameters Found: {'classifier__C': 1, 'tfidf__max_df': 1.0}
Best Cross-Validation F1-Score (Macro): 0.8779


In [30]:
# Get the best pipeline from the grid search
best_pipeline = grid_search_pipeline.best_estimator_

# Make predictions on the test set using the tuned pipeline
y_pred_pipeline = best_pipeline.predict(X_test)

In [31]:
# Evaluate the final tuned model
print("\n--- Final Tuned Pipeline Evaluation ---")
accuracy_pipeline = accuracy_score(y_test, y_pred_pipeline)
print(f"Tuned Pipeline Accuracy: {accuracy_pipeline:.4f}")
print("Tuned Pipeline Classification Report:")
print(classification_report(y_test, y_pred_pipeline))


--- Final Tuned Pipeline Evaluation ---
Tuned Pipeline Accuracy: 0.8938
Tuned Pipeline Classification Report:
                         precision    recall  f1-score   support

Bank account or service       0.87      0.82      0.84      1142
            Credit card       0.85      0.83      0.84      1586
       Credit reporting       0.89      0.87      0.88      2505
        Debt collection       0.88      0.91      0.90      3511
               Mortgage       0.95      0.96      0.95      2984

               accuracy                           0.89     11728
              macro avg       0.89      0.88      0.88     11728
           weighted avg       0.89      0.89      0.89     11728



### Saving the model

In [32]:
import joblib

# Save the best pipeline object to a file
model_filename = 'support_ticket_classifier.joblib'
joblib.dump(best_pipeline, model_filename)

print(f"Model saved as '{model_filename}'")

Model saved as 'support_ticket_classifier.joblib'
